# Experiment 2: Comparing Different ML Models for YouTube Comment Sentiment Analysis

This notebook follows the same MLflow/DagsHub tracking pattern as `experiment_1_baseline_model.ipynb`
(which trained a `RandomForestClassifier` baseline), but trains and logs **several different
classical ML models** on the same `CountVectorizer` features so their metrics can be compared
side-by-side in MLflow:

- Logistic Regression
- Multinomial Naive Bayes
- Linear Support Vector Machine (`LinearSVC`)
- Gradient Boosted Trees (`XGBoost`)

Each model is trained and logged as its own MLflow run under the `Model_Comparison` experiment,
using the same train/test split and vectorizer settings as the baseline for a fair comparison.

In [ ]:
# Install any missing dependencies (uncomment if needed)
# %pip install mlflow dagshub xgboost scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import mlflow
import dagshub

# Same DagsHub-tracked repo used in experiment_1_baseline_model.ipynb
dagshub.init(repo_owner="MitadruMridha05", repo_name="Youtube_Sentiment_Analysis", mlflow=True)

remote_server_uri = "https://dagshub.com/MitadruMridha05/Youtube_Sentiment_Analysis.mlflow"
mlflow.set_tracking_uri(remote_server_uri)

In [ ]:
import numpy as np
import pandas as pd

## Load data

Update `DATA_PATH` to point at your local copy of `youtube_comments_cleaned_final.csv`
(the same file used in `experiment_1_baseline_model.ipynb`).

In [ ]:
DATA_PATH = r"C:\Users\mitad\OneDrive\Documents\coding_maxxing\sentiment_analysing_project\artifacts\youtube_comments_cleaned_final.csv"

df = pd.read_csv(DATA_PATH)
df.shape

In [ ]:
df["Sentiment"].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

## Feature extraction

Same `CountVectorizer` settings as the baseline (`max_features=10000`) so the different
models are compared on identical input features.

In [ ]:
vectorizer = CountVectorizer(max_features=10000)

df["CommentText"] = df["CommentText"].fillna("")
x = vectorizer.fit_transform(df["CommentText"]).toarray()
y = df["Sentiment"]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

x_train.shape, x_test.shape

In [ ]:
mlflow.set_experiment("Model_Comparison")

## Reusable training / logging function

This mirrors the logging pattern from the Random Forest baseline run (tags, params, metrics,
confusion matrix artifact, and the model itself), but is factored into a function so it can be
reused for every model below.

In [ ]:
import os

ARTIFACT_DIR = "artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)


def train_and_log_model(model, model_name, run_name, description, params=None):
    """Train `model`, evaluate it on the held-out test set, and log everything to MLflow."""
    params = params or {}

    with mlflow.start_run(run_name=run_name):
        # Tags
        mlflow.set_tag("model_type", model_name)
        mlflow.set_tag("experiment", "model_comparison")
        mlflow.set_tag("mlflow_runName", run_name)
        mlflow.set_tag("description", description)

        # Vectorizer params (same for every model, logged for traceability)
        mlflow.log_param("vectorizer_type", "CountVectorizer")
        mlflow.log_param("vectorizer_max_features", vectorizer.max_features)

        # Model-specific params
        for param_name, param_value in params.items():
            mlflow.log_param(param_name, param_value)

        # Train
        model.fit(x_train, y_train)

        # Predict
        y_pred = model.predict(x_test)

        # Metrics
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if label not in ["accuracy", "macro avg", "weighted avg"]:
                for metric_name, metric_value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric_name}", metric_value)
            else:
                if isinstance(metrics, dict):
                    for metric_name, metric_value in metrics.items():
                        mlflow.log_metric(f"{label.replace(' ', '_')}_{metric_name}", metric_value)

        # Confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(
            conf_matrix, annot=True, fmt="d", cmap="Blues",
            xticklabels=np.unique(y), yticklabels=np.unique(y),
        )
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix - {model_name}")

        artifact_path = os.path.join(ARTIFACT_DIR, f"confusion_matrix_{model_name}.png")
        plt.savefig(artifact_path, bbox_inches="tight")
        plt.show()

        mlflow.log_artifact(artifact_path, artifact_path="plots")

        # Log the trained model
        mlflow.sklearn.log_model(model, model_name)

        print(f"{model_name} — accuracy: {accuracy:.4f}")

    return accuracy

## Model 1: Logistic Regression

In [ ]:
logreg_params = {"max_iter": 1000, "C": 1.0, "solver": "lbfgs"}

logreg_model = LogisticRegression(**logreg_params, random_state=42)

logreg_accuracy = train_and_log_model(
    logreg_model,
    model_name="LogisticRegression",
    run_name="LogReg_model",
    description="Logistic Regression model for sentiment analysis on YouTube comments.",
    params=logreg_params,
)

## Model 2: Multinomial Naive Bayes

In [ ]:
nb_params = {"alpha": 1.0}

nb_model = MultinomialNB(**nb_params)

nb_accuracy = train_and_log_model(
    nb_model,
    model_name="MultinomialNB",
    run_name="NaiveBayes_model",
    description="Multinomial Naive Bayes model for sentiment analysis on YouTube comments.",
    params=nb_params,
)

## Model 3: Linear Support Vector Machine

In [ ]:
svm_params = {"C": 1.0, "max_iter": 5000}

svm_model = LinearSVC(**svm_params, random_state=42)

svm_accuracy = train_and_log_model(
    svm_model,
    model_name="LinearSVC",
    run_name="SVM_model",
    description="Linear SVM model for sentiment analysis on YouTube comments.",
    params=svm_params,
)

## Model 4: XGBoost

`XGBClassifier` expects labels starting at 0, so the `-1 / 0 / 1` sentiment labels
are remapped to `0 / 1 / 2` just for this model, then mapped back for the report.

In [ ]:
xgb_params = {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.1}

# XGBoost needs 0-indexed integer labels
label_map = {-1: 0, 0: 1, 1: 2}
inverse_label_map = {v: k for k, v in label_map.items()}

y_train_xgb = y_train.map(label_map)
y_test_xgb = y_test.map(label_map)

xgb_model = XGBClassifier(
    **xgb_params,
    objective="multi:softmax",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42,
)

with mlflow.start_run(run_name="XGBoost_model"):
    mlflow.set_tag("model_type", "XGBClassifier")
    mlflow.set_tag("experiment", "model_comparison")
    mlflow.set_tag("mlflow_runName", "XGBoost_model")
    mlflow.set_tag("description", "XGBoost model for sentiment analysis on YouTube comments.")

    mlflow.log_param("vectorizer_type", "CountVectorizer")
    mlflow.log_param("vectorizer_max_features", vectorizer.max_features)
    for param_name, param_value in xgb_params.items():
        mlflow.log_param(param_name, param_value)

    xgb_model.fit(x_train, y_train_xgb)
    y_pred_xgb = xgb_model.predict(x_test)

    # Map predictions back to the original -1 / 0 / 1 labels for reporting
    y_pred = pd.Series(y_pred_xgb).map(inverse_label_map)

    xgb_accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", xgb_accuracy)

    classification_rep = classification_report(y_test, y_pred, output_dict=True)
    for label, metrics in classification_rep.items():
        if label not in ["accuracy", "macro avg", "weighted avg"]:
            for metric_name, metric_value in metrics.items():
                mlflow.log_metric(f"{label}_{metric_name}", metric_value)

    conf_matrix = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        conf_matrix, annot=True, fmt="d", cmap="Blues",
        xticklabels=np.unique(y), yticklabels=np.unique(y),
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix - XGBoost")

    artifact_path = os.path.join(ARTIFACT_DIR, "confusion_matrix_XGBoost.png")
    plt.savefig(artifact_path, bbox_inches="tight")
    plt.show()

    mlflow.log_artifact(artifact_path, artifact_path="plots")
    mlflow.sklearn.log_model(xgb_model, "XGBClassifier")

    print(f"XGBoost — accuracy: {xgb_accuracy:.4f}")

## Compare all models

In [ ]:
results = pd.DataFrame({
    "model": ["LogisticRegression", "MultinomialNB", "LinearSVC", "XGBoost"],
    "accuracy": [logreg_accuracy, nb_accuracy, svm_accuracy, xgb_accuracy],
}).sort_values("accuracy", ascending=False).reset_index(drop=True)

results

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=results, x="model", y="accuracy", hue="model", legend=False, palette="Blues_d")
plt.ylim(0, 1)
plt.title("Model Comparison — Test Accuracy")
plt.ylabel("Accuracy")
plt.xlabel("")
plt.show()

## Next steps

- Compare full precision/recall/F1 per class (logged to MLflow for every run) rather than just accuracy.
- Try TF-IDF instead of `CountVectorizer`, or `n-gram` features, on the strongest model(s) above.
- Tune hyperparameters (e.g. `GridSearchCV` / `RandomizedSearchCV`) for the top performer.
- Register the best model in the MLflow Model Registry for deployment.